# ex04 · 池化层（对应教材 6.5 汇聚层）

> **做题流程**：从零实现 maxpool / avgpool，再理解池化的作用。
> **做完再看** `solutions/ex04-答案.md`。
>
> 难度标记：🌱 基础（预测+验证）｜🔧 变式（改动观察）｜🚀 挑战（闭卷复现）

## 题 1 🔧 从零实现池化（TODO 6.6）

池化 = 在固定窗口上取 max 或 avg，窗口不重叠。补全 pool2d。
（手算对照：X=[[0,1,2],[3,4,5],[6,7,8]]、窗口 2×2 → max=4、avg=2）

In [1]:
import torch
from torch import nn

def pool2d(X, pool_size, mode='max'):
    # TODO 6.6: 按 pool_size 不重叠地取窗口，mode='max' 取最大值、'avg' 取平均值
    # 提示: 双重循环，窗口 = X[i*ph:(i+1)*ph, j*pw:(j+1)*pw]
    ph, pw = pool_size
    yh = int(X.shape[0] / ph)
    yw = int(X.shape[1] / pw)
    Y = torch.zeros(yh, yw)
    if mode == 'max':
        for i in range(yh):
            for j in range(yw):
                Y[i][j] = X[i*ph : (i+1)*ph, j*pw : (j+1)*pw].max()
    if mode == 'avg':
        for i in range(yh):
            for j in range(yw):
                Y[i][j] = X[i*ph : (i+1)*ph, j*pw : (j+1)*pw].mean()

    return Y

In [2]:
try:
    X = torch.tensor([[0.,1.,2.],[3.,4.,5.],[6.,7.,8.]])
    assert pool2d(X, (2, 2), 'max').item() == 4.0
    assert abs(pool2d(X, (2, 2), 'avg').item() - 2.0) < 1e-6
    print('✓ pool2d 正确: max=4, avg=2')

    # 与 nn.MaxPool2d / nn.AvgPool2d 对比（4×4 输入）
    X4 = torch.arange(16, dtype=torch.float32).reshape(4, 4)
    assert torch.equal(pool2d(X4, (2, 2), 'max'), nn.MaxPool2d(2)(X4.reshape(1,1,4,4)).reshape(2,2))
    print('✓ 与 nn.MaxPool2d 一致')
except NotImplementedError as e:
    print(f'⚠ {e}')
except AssertionError as e:
    print(f'✗ {e}')

✓ pool2d 正确: max=4, avg=2
✓ 与 nn.MaxPool2d 一致


## 题 2 🌱 池化为什么没有可学习参数（问答）

先回答：池化层和卷积层最本质的区别？池化为什么能「下采样」却没有参数可学？

In [3]:
# 验证：池化层没有任何参数
pool_max = nn.MaxPool2d(2)
print('MaxPool2d 参数量:', sum(p.numel() for p in pool_max.parameters()))
print('MaxPool2d 的 state_dict:', pool_max.state_dict())

# 多通道池化逐通道独立做，不改变通道数
X2 = torch.arange(64, dtype=torch.float32).reshape(2, 2, 4, 4)
print('多通道池化输出形状:', list(nn.MaxPool2d(2)(X2).shape))

MaxPool2d 参数量: 0
MaxPool2d 的 state_dict: OrderedDict()
多通道池化输出形状: [2, 2, 2, 2]


## 小结与面试衔接

- 池化：固定窗口取 max/avg，无参数、逐通道独立
- 作用：下采样（减小特征图）、平移不变性、降计算量
- 与卷积的区别：卷积有可学习核，池化是写死的聚合